# Data preprocessing

This notebook describes the data preprocessing steps for the dataset of user interactions with coping challenges as described in Chapter 2 of the thesis. Additionally, it includes the code for correcting incorrect data due to technical issues (Chapter 2), as well as the code for analyzing adherence between agency and non-agency conditions (Chapter 4).

**Author:** Shirley Li \
**Date:** July 2026

In [1]:
import pandas as pd
import numpy as np
import ast
import os
import utils.bayesian_analysis as ba
import re
from pathlib import Path
import pyperclip

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


### Load data

In [ ]:
data_folder = '..\\data\\'

challenge_completions = pd.read_csv(os.path.join(data_folder, 'challenge_completions.csv'), index_col=0)
participant_conditions = pd.read_csv(os.path.join(data_folder, 'participant_conditions.csv'), index_col=0)
postquestions = pd.read_csv(os.path.join(data_folder, 'postquestions.csv'), index_col=0)
prequestions = pd.read_csv(os.path.join(data_folder, 'prequestions.csv'), index_col=0)
presented_challenges = pd.read_csv(os.path.join(data_folder, 'presented_challenges.csv'), index_col=0)
challenges_sets = pd.read_csv(os.path.join(data_folder, 'challenges_sets.csv'), index_col=0)
photo_df = pd.read_csv(os.path.join(data_folder, 'foto_finalQuestionnaire.csv'), index_col=0)

### Check and remove duplicates

In [4]:
# Each participant should have one row per day in the postquestions, with consistent values for 'Likedness', 'difficulty', 'time_spent', and 'usefulness' across rows with the same 'random_id' and 'day_number'. We will check for any inconsistencies in these columns.
keys = ['random_id', 'day_number']
postquestions_columns = ['random_id', 'day_number', 'Likedness', 'difficulty', 'time_spent', 'usefulness']

# Only keep relevant columns
postquestions = postquestions[postquestions_columns] 
mismatched = postquestions.groupby(keys).filter(lambda x: x.nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")

# We keep the last entry for each duplicate key combination, assuming it is the most recent and therefore the most accurate reflection of the participant's experience for that day.
postquestions = postquestions.drop_duplicates(subset=keys, keep='last')

Found 8 rows with inconsistent ratings across keys.


In [5]:
keys = ['random_id', 'day_number']
presented_challenges_columns = ['random_id', 'day_number', 'challenge_id']
# presented_challenges = presented_challenges[presented_challenges_columns]

mismatched = presented_challenges.groupby(keys).filter(lambda x: x[presented_challenges_columns].nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")
unique_mismatched_keys = mismatched[keys].drop_duplicates()
presented_challenges = presented_challenges.drop_duplicates(subset=keys, keep='last')

Found 5702 rows with inconsistent ratings across keys.


Challenge completions should not contain any duplicates where the same participant has multiple completed challenge_ids for the same day


In [6]:
keys = ['random_id', 'day_number']
challenge_completions_columns = ['random_id', 'day_number', 'challenge_id', 'challenge_category', 'submission_type']
challenge_completions = challenge_completions[challenge_completions_columns]

mismatched = challenge_completions.groupby(keys).filter(lambda x: x.nunique().max() > 1)

print(f"Found {len(mismatched)} rows with inconsistent ratings across keys.")

Found 0 rows with inconsistent ratings across keys.


The number of presented challenges should be the same as the number of filled in prequestions

In [7]:
print(f"Number of presented challenges: {len(presented_challenges)}")
print(f"Number of prequestions: {len(prequestions)}")

merged = pd.merge(presented_challenges, prequestions, on=['random_id', 'day_number'], how='outer', indicator=True)
mismatched = merged[merged['_merge'] != 'both']
print(f"Found {len(mismatched)} rows with mismatched presented challenges and prequestions.")


Number of presented challenges: 5796
Number of prequestions: 5800
Found 4 rows with mismatched presented challenges and prequestions.


### Extract completed challenge information
We merge the completed challenges with the postquestions to obtain the ratings for the challenges

In [8]:
keys = ['random_id', 'day_number', 'challenge_id']
completed_challenges = challenge_completions[['random_id', 'day_number', 'challenge_id', 'challenge_category', 'submission_type']].copy()

# Check for duplicates in the completed challenges and only keep last entry for each duplicate key combination, 
# assuming it is the most recent and therefore the most accurate reflection of the participant's experience for that day.
mismatched = completed_challenges[completed_challenges.duplicated(keys, keep=False)]
print(f"Number of duplicate challenge entries (keep last): {len(mismatched)}")
completed_challenges = completed_challenges.drop_duplicates(subset=keys, keep='last')

completed_challenges = completed_challenges.merge(postquestions, on=['random_id', 'day_number'], how='left')
print("Number of challenge ratings:", len(completed_challenges))
completed_challenges.head()

challenge_info = (
    completed_challenges[['challenge_id', 'challenge_category', 'submission_type']]
    .drop_duplicates()
    .rename(columns={'challenge_category': 'category'})
)

cat_to_id = {c: i for i, c in enumerate(sorted(challenge_info['category'].unique()))}
challenge_info['category_id'] = challenge_info['category'].map(cat_to_id)

mean_challenge_scores = (
    completed_challenges
    .groupby('challenge_id')[['Likedness', 'difficulty', 'time_spent', 'usefulness']]
    .mean()
    .rename(columns={'Likedness': 'likedness'})
)

challenge_info = challenge_info.merge(mean_challenge_scores, on='challenge_id', how='left')
challenge_info = challenge_info.sort_values(by='challenge_id').reset_index(drop=True)
challenge_info.to_csv(os.path.join(data_folder, 'challenge_info.csv'), index=False)

Number of duplicate challenge entries (keep last): 807
Number of challenge ratings: 3784


In [9]:
print("Number of challenges:", len(challenge_info))

Number of challenges: 103


## Extract relevant data
We first define a helper function to get the within condition based on the day and the order of within conditions

In [10]:
def get_condition(row):
    day = row['day_number']
    order_list = row['within_order']
    if day <= 5:
        return 'w0'
    idx = ((day - 1) // 5) - 1
    return order_list[idx]

Obtain relevant data that describes users' state, the challenge recommendation, and the challenge completion information for each day of the study. This data will be for all implementation and analysis of the thesis.

In [11]:
prequestions_cols = ['random_id', 'day_number', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY']
df_samples = prequestions[prequestions_cols].copy()

# Merge participant conditions to get between_condition and within_order
df_samples = df_samples.merge(participant_conditions[['random_id', 'between_condition', 'within_order']], on='random_id', how='left')
df_samples['within_order'] = df_samples['within_order'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
df_samples['within_condition'] = df_samples.apply(lambda row: get_condition(row), axis=1)

# Save initial states distribution for Day 1 for all users, regardless of their condition
initial_states = df_samples[df_samples['day_number'] == 1]
initial_states[['random_id', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY']].to_csv(os.path.join(data_folder, 'initial_states_distribution.csv'), index=False)

# Merge with recommended challenges to get challenge_id for each sample
recommended_challenges = challenges_sets[['random_id', 'day_number', 'recommended_challenge_id']].copy()
df_samples = df_samples.merge(recommended_challenges, on=['random_id', 'day_number'], how='left')

# Mark everything in the completed df as 1
completed_challenges['completed'] = 1

# Merge challenge completions
df_samples = df_samples.merge(
    completed_challenges[['random_id', 'day_number', 'challenge_id', 'completed', 'Likedness', 'difficulty', 'time_spent', 'usefulness']], 
    on=['random_id', 'day_number'], 
    how='left'
)

deviations = df_samples[
    df_samples['challenge_id'].notna() &
    (df_samples['challenge_id'] != df_samples['recommended_challenge_id'])
]
# Deviations should only be possible in agency conditions
assert len(deviations[deviations['within_condition'].isin(['w0','w1', 'w2'])]) == 0, "There are samples in the non-agency conditions where the presented challenge does not match the recommended challenge."

# Fill not completed challenges accordingly
df_samples['completed'] = df_samples['completed'].fillna(0).astype(int)
df_samples['challenge_id'] = df_samples['challenge_id'].fillna(df_samples['recommended_challenge_id'])

# Set postquestions to NaN for non-completed challenges
post_cols = ['Likedness', 'difficulty', 'time_spent', 'usefulness']
for col in post_cols:
    df_samples.loc[df_samples['completed'] == 0, col] = np.nan

df_samples = df_samples.sort_values(by=['random_id', 'day_number'])
df_samples['PAY_next'] = df_samples.groupby('random_id')['PAY'].shift(-1)

# remove rows where next state is NaN (i.e., last day for each user)
# df_samples = df_samples.dropna(subset=['TIR_next', 'MOT_next', 'GOOD_next', 'TIME_Q_next', 'PAY_next'])
df_samples = df_samples.rename(columns={"Likedness" : "likedness", "random_id": "user_id"})

final_columns = [
        'user_id', 'day_number', 'TIR', 'MOT', 'GOOD', 'TIME_Q', 'PAY',
        'PAY_next', 'recommended_challenge_id',
        'challenge_id', 'completed', 'likedness', 'difficulty', 'time_spent', 'usefulness', 'within_condition', 'between_condition'
    ]
df_samples = df_samples[final_columns]

# Remove rows where challenge_id is NaN, as these represent the day where the data was pulled and no challenge was presented yet
df_samples.dropna(subset=['challenge_id'], inplace=True)

# Merge challenge information into samples
df_samples = df_samples.merge(challenge_info[['challenge_id', 'category', 'category_id', 'submission_type']], on='challenge_id', how='left')

print("Total number of samples:", len(df_samples))
print("Total number of users:", df_samples['user_id'].nunique())

Total number of samples: 5796
Total number of users: 317


## Correct incompleted photo challenges
Due to a technical issue in the application, participants had trouble uploading photos for these type of challenges, leading to incomplete/wrong entries in the data. This section describes how we handle these cases and correct the data accordingly.

#### Mean ratings for photo challenges
These will be used to fill in the missing ratings for the photo challenges that were not marked as completed due to technical issues.

In [12]:
mean_vals_photo = df_samples[(df_samples['submission_type'] == 'image') & (df_samples['completed'] == 1)].groupby('challenge_id')[["likedness", "difficulty", "usefulness", "time_spent"]].mean()
mean_vals_photo["count"] = df_samples[(df_samples['submission_type'] == 'image') & (df_samples['completed'] == 1)].groupby('challenge_id').size()

#### Correct photo challenges

In [13]:
photo_counts = (
    df_samples[df_samples['submission_type'] == 'image']
    .groupby('user_id')['completed']
    .agg(
        photos_completed='sum',
        photos_not_completed=lambda x: (x == 0).sum()
    )
    .astype(int)
)

photo_df_corrections = (
    photo_counts
    .merge(photo_df[['random_id', 'photo_challenge_n']], left_index=True, right_on='random_id', how='left')
    .fillna({'photo_challenge_n': 0})
    .assign(photo_challenge_n=lambda df: df['photo_challenge_n'].astype(int))
)

total = photo_df_corrections['photos_completed'] + photo_df_corrections['photos_not_completed']

photo_df_corrections['n_to_adjust'] = np.minimum(photo_df_corrections['photo_challenge_n'], photo_df_corrections['photos_not_completed']) 

# for every user, find probability that the not-completed photo, was actually completed, 
# i.e. photo_challenge_n / photos_not_completed, but only for users where photos_not_completed > 0 to avoid division by zero
photo_df_corrections['prob_photo_completed'] = np.where(
    photo_df_corrections['photos_not_completed'] > 0,
    photo_df_corrections['n_to_adjust'] / photo_df_corrections['photos_not_completed'],
    0)

# for every user, adjust the samples according to the probability found above
# i.e. for every sample where Type is Photo and completed is 0, we set completed to 1 with the probability found above for that user
# additionally, impute mean values for likedness, difficulty, usefulness, and time_spent for the newly completed photos
df_samples_adjusted = df_samples.copy()
rng = np.random.default_rng(66)

for idx, row in photo_df_corrections.iterrows():
    user_id = row['random_id']
    prob_completed = row['prob_photo_completed']
    n_to_adjust = row['n_to_adjust'].astype(int)
    
    # Get indices of samples for this user where Type is Photo and completed is 0
    mask = (df_samples_adjusted['user_id'] == user_id) & (df_samples_adjusted['submission_type'] == 'image') & (df_samples_adjusted['completed'] == 0)
    indices_to_adjust = df_samples_adjusted[mask].index
    selected_indices = indices_to_adjust[rng.random(len(indices_to_adjust)) < prob_completed]

    # select maximum of n_to_adjust indices to adjust
    if len(selected_indices) > n_to_adjust:
        selected_indices = rng.choice(selected_indices, size=n_to_adjust, replace=False)
  
    # For each of these samples, we will set completed to 1 with the probability found above for that user, and impute mean values for likedness, difficulty, usefulness, and time_spent
    for idx in selected_indices:
        challenge_id = df_samples_adjusted.at[idx, 'challenge_id']
        df_samples_adjusted.at[idx, 'completed'] = 1
        df_samples_adjusted.at[idx, 'likedness'] = mean_vals_photo.loc[challenge_id, 'likedness']
        df_samples_adjusted.at[idx, 'difficulty'] = mean_vals_photo.loc[challenge_id, 'difficulty']
        df_samples_adjusted.at[idx, 'usefulness'] = mean_vals_photo.loc[challenge_id, 'usefulness']
        df_samples_adjusted.at[idx, 'time_spent'] = mean_vals_photo.loc[challenge_id, 'time_spent']


#### Save
We save all samples from both agency and non-agency states. During further data processing into (s, a, r, s') tuples, we will filter out the samples that are not relevant for the analysis.

In [14]:
df_samples_adjusted.to_csv(os.path.join(data_folder, 'processed_samples_corrected_full.csv'), index=False)

print("Final number of total samples:", len(df_samples_adjusted))
print("Final number of users:", df_samples_adjusted['user_id'].nunique())

Final number of total samples: 5796
Final number of users: 317


### Compare adherence in agency vs. non-agency conditions
For simulating the effect of having choice on challenge completion, we compare the adherence in the agency conditions (w3, w4, w5) with the non-agency conditions (w0, w1, w2) using a Bayesian paired t-test. 

In [15]:
conditions_to_keep = ['w0', 'w1', 'w2']
df_final_adjusted = df_samples_adjusted[df_samples_adjusted['within_condition'].isin(conditions_to_keep)].copy()
df_agency_adjusted = df_samples_adjusted[df_samples_adjusted['within_condition'].isin(['w3', 'w4'])].copy() 

# Per-person completion rates
agency_rates = (
    df_agency_adjusted
    .groupby("user_id")["completed"]
    .mean()
)

control_rates = (
    df_final_adjusted
    .groupby("user_id")["completed"]
    .mean()
)

paired = pd.concat(
    [agency_rates, control_rates],
    axis=1,
    keys=["agency", "control"]
).dropna()

# Per-person differences
paired["diff"] = paired["agency"] - paired["control"]

diffs = paired["diff"].values

results_ba_agency = ba.paired_t_test(diffs, verbose=True)
mean_diff, std_diff = results_ba_agency["mean"].values[0], results_ba_agency["std"].values[0]
pyperclip.copy(ba.to_latex_body(results_ba_agency))


Posterior probability improvement > 0: 0.9900
              mean     sd  hdi_2.5%  hdi_97.5%
improvement  0.031  0.013     0.006      0.055
effect_size  0.157  0.067     0.027      0.287


Write to config file

In [16]:
path = Path("config.py")
text = path.read_text()
text = re.sub(r'agency_bias\s*=\s*\(([\d.]+),\s*([\d.]+)\)', f'agency_bias = ({mean_diff:.3f}, {std_diff:.3f})', text)
path.write_text(text)    

988